# ⚙️ Notebook 2 — Feature Engineering & Data Preprocessing

## What this notebook does
- Handles missing values with domain-aware imputation
- Encodes categorical variables
- Engineers new features that improve model performance
- Applies SMOTE to handle class imbalance
- Saves processed data ready for modelling

**Input:** `data/raw/transactions.csv` 
**Output:** `data/processed/features_train.csv`, `data/processed/features_test.csv`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/transactions.csv')
print(f'Raw shape: {df.shape}')

## 1. Missing Value Imputation

> **Domain logic:** Unknown countries are a risk signal in themselves — we impute as 'Unknown' rather than dropping. Amounts we impute with the median per channel (less biased than global median).

In [ ]:
# Country: Unknown is informative — keep as category
df['country'] = df['country'].fillna('Unknown')

# Amount: median per channel (domain-aware imputation)
channel_medians = df.groupby('channel')['amount'].median()
df['amount'] = df.apply(
    lambda r: channel_medians[r['channel']] if pd.isna(r['amount']) else r['amount'],
    axis=1
)

print('Missing values after imputation:')
print(df.isnull().sum())
print('\n✓ No missing values remain')

## 2. Feature Engineering

> These are the features that separate a junior from a senior analyst. We create new variables that capture domain knowledge the raw data doesn't express directly.

In [ ]:
# ── Time-based risk features ────────────────────────────────────────────
df['is_night']       = ((df['hour'] >= 23) | (df['hour'] <= 2)).astype(int)
df['is_peak_fraud']  = ((df['hour'] >= 22) | (df['hour'] <= 3)).astype(int)

# ── Amount-based features ───────────────────────────────────────────────
df['amount_log']         = np.log1p(df['amount'])
df['is_high_amount']     = (df['amount'] > 1500).astype(int)
df['is_round_amount']    = (df['amount'] % 100 == 0).astype(int)  # round amounts = suspicious

# ── Velocity risk feature ────────────────────────────────────────────────
df['high_velocity']      = (df['transactions_last_24h'] > 5).astype(int)

# ── Customer tenure feature ─────────────────────────────────────────────
df['is_new_customer']    = (df['days_as_customer'] < 30).astype(int)
df['customer_tenure_log']= np.log1p(df['days_as_customer'])

# ── Geography risk ───────────────────────────────────────────────────────
high_risk_countries = ['Nigeria','Romania','Unknown']
df['high_risk_country']  = df['country'].isin(high_risk_countries).astype(int)

# ── Composite risk score (interpretable for dashboard) ──────────────────
df['risk_score'] = (
    df['is_night']         * 2.0 +
    df['high_risk_country']* 3.0 +
    df['is_high_amount']   * 1.5 +
    df['high_velocity']    * 2.0 +
    df['is_new_customer']  * 1.5 +
    (df['channel'] == 'online').astype(int) * 1.0
)

print('New features created:')
new_features = ['is_night','is_peak_fraud','amount_log','is_high_amount',
                'is_round_amount','high_velocity','is_new_customer',
                'customer_tenure_log','high_risk_country','risk_score']
print(df[new_features].describe().round(2))

## 3. Encoding & Feature Selection

In [ ]:
# Encode categoricals
le_country = LabelEncoder()
le_channel = LabelEncoder()
df['country_enc'] = le_country.fit_transform(df['country'])
df['channel_enc'] = le_channel.fit_transform(df['channel'])

# Final feature list
FEATURES = [
    'amount', 'amount_log', 'hour', 'transactions_last_24h',
    'days_as_customer', 'customer_tenure_log', 'is_night', 'is_peak_fraud',
    'is_high_amount', 'is_round_amount', 'high_velocity', 'is_new_customer',
    'high_risk_country', 'risk_score', 'country_enc', 'channel_enc'
]
TARGET = 'is_fraud'

X = df[FEATURES]
y = df[TARGET]

print(f'Feature matrix shape: {X.shape}')
print(f'Target distribution: {y.value_counts().to_dict()}')

## 4. Train/Test Split + SMOTE

> We apply SMOTE (Synthetic Minority Oversampling Technique) **only on the training set** — never on test data. This is a common mistake that leaks information and inflates evaluation metrics artificially.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {X_train.shape[0]:,} | Fraud in train: {y_train.sum():,} ({y_train.mean()*100:.1f}%)')
print(f'Test size:  {X_test.shape[0]:,}  | Fraud in test:  {y_test.sum():,} ({y_test.mean()*100:.1f}%)')

# Apply SMOTE on training data only
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'\nAfter SMOTE — Train size: {X_train_sm.shape[0]:,}')
print(f'Class balance: {pd.Series(y_train_sm).value_counts().to_dict()}')

# Save processed data
train_df = pd.DataFrame(X_train_sm, columns=FEATURES)
train_df['is_fraud'] = y_train_sm.values
train_df.to_csv('../data/processed/features_train.csv', index=False)

test_df = pd.DataFrame(X_test, columns=FEATURES)
test_df['is_fraud'] = y_test.values
test_df.to_csv('../data/processed/features_test.csv', index=False)

print('\n✓ Processed data saved to data/processed/')